# A seleção que mais marcou em cada Copa

## Task 3: a seleção que mais marcou em cada Copa

## A pergunta do time de analytics

**Qual seleção mais marcou gols em cada Copa?**

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-rankings")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta.

In [2]:
# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")

In [32]:
# Qual seleção mais marcou gols em cada Copa?
# qntd total de gols de cada selecao em cada copa
# qual selecao fez max(gols) em cd copa

spark.sql("SELECT ano, selecao, gols_pro, saldo FROM resumo ORDER BY ano").show(5)

+----+-------------+--------+-----+
| ano|      selecao|gols_pro|saldo|
+----+-------------+--------+-----+
|1930|     Paraguay|       1|   -2|
|1930|         Peru|       1|   -3|
|1930|United States|       7|    1|
|1930|       France|       4|    1|
|1930|   Yugoslavia|       7|    0|
+----+-------------+--------+-----+
only showing top 5 rows


In [34]:
spark.sql("""
SELECT ano, selecao, gols_pro, saldo 
FROM resumo 
ORDER BY ano ASC, gols_pro DESC, saldo DESC
""").show(15)

+----+-------------+--------+-----+
| ano|      selecao|gols_pro|saldo|
+----+-------------+--------+-----+
|1930|    Argentina|      18|    9|
|1930|      Uruguay|      15|   12|
|1930|United States|       7|    1|
|1930|   Yugoslavia|       7|    0|
|1930|       Brazil|       5|    3|
|1930|        Chile|       5|    2|
|1930|       France|       4|    1|
|1930|       Mexico|       4|   -9|
|1930|      Romania|       3|   -2|
|1930|     Paraguay|       1|   -2|
|1930|         Peru|       1|   -3|
|1930|      Belgium|       0|   -4|
|1930|      Bolivia|       0|   -8|
|1934|        Italy|      12|    9|
|1934|      Germany|      11|    3|
+----+-------------+--------+-----+
only showing top 15 rows


In [37]:
spark.sql("""
SELECT ano, selecao, MAX(gols_pro)
FROM resumo
GROUP BY ano, selecao
ORDER BY ano
""").show(15)

+----+--------------+-------------+
| ano|       selecao|max(gols_pro)|
+----+--------------+-------------+
|1930|         Chile|            5|
|1930|        France|            4|
|1930|     Argentina|           18|
|1930|       Uruguay|           15|
|1930|        Brazil|            5|
|1930|      Paraguay|            1|
|1930| United States|            7|
|1930|    Yugoslavia|            7|
|1930|       Romania|            3|
|1930|       Belgium|            0|
|1930|          Peru|            1|
|1930|        Mexico|            4|
|1930|       Bolivia|            0|
|1934|Czechoslovakia|            9|
|1934|         Spain|            4|
+----+--------------+-------------+
only showing top 15 rows


## Por que o groupBy não basta

O `groupBy` junta as linhas de cada Copa numa só, e a seleção se perde. Uma **window function** calcula olhando o
grupo, mas **mantém todas as linhas**, com o resultado numa coluna nova.

| Parte | Pergunta | No nosso caso |
|---|---|---|
| `partitionBy` | quais linhas formam um grupo? | as seleções de uma **mesma Copa** |
| `orderBy` | em que ordem, dentro do grupo? | dos **mais gols** para os menos |

## A posição de cada seleção na sua Copa

`row_number()` numera as linhas de cada grupo: 1, 2, 3... O saldo e o nome entram no `orderBy` como desempate.

In [41]:
from pyspark.sql import Window

por_copa_mais_gols = Window.partitionBy("ano").orderBy(F.desc("gols_pro"), F.desc("saldo"), "selecao")

gols_selecoes = resumo.select("ano", "selecao", "gols_pro", "saldo")
ranking = gols_selecoes.withColumn("posicao", F.row_number().over(por_copa_mais_gols))

ranking.show(15)

+----+-------------+--------+-----+-------+
| ano|      selecao|gols_pro|saldo|posicao|
+----+-------------+--------+-----+-------+
|1930|    Argentina|      18|    9|      1|
|1930|      Uruguay|      15|   12|      2|
|1930|United States|       7|    1|      3|
|1930|   Yugoslavia|       7|    0|      4|
|1930|       Brazil|       5|    3|      5|
|1930|        Chile|       5|    2|      6|
|1930|       France|       4|    1|      7|
|1930|       Mexico|       4|   -9|      8|
|1930|      Romania|       3|   -2|      9|
|1930|     Paraguay|       1|   -2|     10|
|1930|         Peru|       1|   -3|     11|
|1930|      Belgium|       0|   -4|     12|
|1930|      Bolivia|       0|   -8|     13|
|1934|        Italy|      12|    9|      1|
|1934|      Germany|      11|    3|      2|
+----+-------------+--------+-----+-------+
only showing top 15 rows


In [43]:
mais_gols_por_copa = ranking.filter(F.col("posicao") == 1).orderBy("ano")
mais_gols_por_copa.select("ano", "selecao", "gols_pro", "saldo").show(22)

+----+------------+--------+-----+
| ano|     selecao|gols_pro|saldo|
+----+------------+--------+-----+
|1930|   Argentina|      18|    9|
|1934|       Italy|      12|    9|
|1938|     Hungary|      15|   10|
|1950|      Brazil|      22|   16|
|1954|     Hungary|      27|   17|
|1958|      France|      23|    8|
|1962|      Brazil|      14|    9|
|1966|    Portugal|      17|    9|
|1970|      Brazil|      19|   12|
|1974|      Poland|      16|   11|
|1978|   Argentina|      15|   11|
|1982|      France|      16|    4|
|1986|   Argentina|      14|    9|
|1990|West Germany|      15|   10|
|1994|      Sweden|      15|    7|
|1998|      France|      15|   13|
|2002|      Brazil|      18|   14|
|2006|     Germany|      14|    8|
|2010|     Germany|      16|   11|
|2014|     Germany|      18|   14|
|2018|     Belgium|      16|   10|
|2022|      France|      16|    8|
+----+------------+--------+-----+



## A mesma pergunta em SQL, com `QUALIFY`

`OVER (PARTITION BY ... ORDER BY ...)` é a janela; o `QUALIFY` filtra pelo resultado dela.

In [45]:
spark.sql("""
    SELECT ano, selecao, gols_pro
    FROM resumo
    QUALIFY ROW_NUMBER() OVER(PARTITION BY ano ORDER BY gols_pro DESC, saldo DESC, selecao) = 1
    ORDER BY ano
""").show(25)

+----+------------+--------+
| ano|     selecao|gols_pro|
+----+------------+--------+
|1930|   Argentina|      18|
|1934|       Italy|      12|
|1938|     Hungary|      15|
|1950|      Brazil|      22|
|1954|     Hungary|      27|
|1958|      France|      23|
|1962|      Brazil|      14|
|1966|    Portugal|      17|
|1970|      Brazil|      19|
|1974|      Poland|      16|
|1978|   Argentina|      15|
|1982|      France|      16|
|1986|   Argentina|      14|
|1990|West Germany|      15|
|1994|      Sweden|      15|
|1998|      France|      15|
|2002|      Brazil|      18|
|2006|     Germany|      14|
|2010|     Germany|      16|
|2014|     Germany|      18|
|2018|     Belgium|      16|
|2022|      France|      16|
+----+------------+--------+



## Para consultar depois

- **`rank` e `dense_rank`:** também numeram as linhas, mas tratam empates de outro jeito. Com dois empatados em
  segundo, `rank` dá 1, 2, 2, 4 (pula a posição) e `dense_rank` dá 1, 2, 2, 3. O `row_number` sempre dá 1, 2, 3, 4.
- **`lag` e `lead`:** pegam o valor da linha anterior ou da próxima dentro do grupo, por exemplo os gols da seleção
  na Copa anterior (janela por seleção, ordenada por ano).
- **Acumulado:** `F.sum(...)` sobre uma janela com `rowsBetween(Window.unboundedPreceding, Window.currentRow)` soma do
  início do grupo até a linha atual.
- **Janela sem `orderBy`** vê o grupo inteiro: serve para colocar o total do grupo em cada linha e calcular
  percentuais.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [46]:
spark.stop()